In [11]:
# 根据 operational form 生成时间区间表
import pandas as pd
from datetime import timedelta
flare_event_df = pd.read_csv("./data/flare_records.csv")

# 转换 `Flare Start Time` 列为 datetime 类型
flare_event_df['Flare Start Time'] = pd.to_datetime(flare_event_df['Flare Start Time'])

# 筛选 `Flare Level >= 20` 的数据
flare_event_df_filtered = flare_event_df[flare_event_df['Flare Level'] >= 20]

# 按 `NOAA_AR` 和 `HARPNUM` 分组
grouped = flare_event_df_filtered.groupby(['NOAA_AR', 'HARPNUM'])

# 存储合并后的结果
merged_intervals = []

for (noaa_ar, harpnum), group in grouped:
    # 对每个 `HARPNUM` 按 `Flare Start Time` 排序
    group = group.sort_values(by='Flare Start Time')
    
    # 创建区间：Flare Start Time - 24h 到 Flare Start Time
    intervals = []
    for _, row in group.iterrows():
        start_time = row['Flare Start Time'] - timedelta(hours=24)
        end_time = row['Flare Start Time']
        intervals.append((start_time, end_time))
    
    # 合并区间（若有重叠或相邻的区间，则合并）
    merged = []
    intervals.sort()
    current_start, current_end = intervals[0]
    
    for start, end in intervals[1:]:
        if start <= current_end:  # 如果区间重叠或相邻
            current_end = max(current_end, end)  # 合并区间
        else:
            merged.append((current_start, current_end))  # 保存上一个区间
            current_start, current_end = start, end  # 更新为新区间
    
    # 不要忘记保存最后一个区间
    merged.append((current_start, current_end))
    
    # 保存合并后的区间
    for start, end in merged:
        merged_intervals.append({
            'HARPNUM': harpnum,
            'Interval Start': start,
            'Interval End': end,
            'NOAA_AR': noaa_ar
        })

# 创建合并后的 DataFrame
result_df = pd.DataFrame(merged_intervals)

# 显示结果
print(result_df)

# 保存结果为 CSV 文件
result_df.to_csv('./data/operational_form.csv', index=False)

      HARPNUM      Interval Start        Interval End  NOAA_AR
0          49 2010-06-12 05:30:00 2010-06-13 05:30:00    11079
1          51 2010-06-11 03:57:00 2010-06-12 03:57:00    11080
2          51 2010-06-12 07:05:00 2010-06-13 07:05:00    11080
3          54 2010-06-11 00:30:00 2010-06-14 00:44:00    11081
4          86 2010-07-07 22:03:00 2010-07-09 19:26:00    11087
...       ...                 ...                 ...      ...
1074     8602 2022-09-17 10:45:00 2022-09-21 05:12:00    13102
1075     8602 2022-09-24 02:51:00 2022-09-25 20:54:00    13102
1076     8599 2022-09-15 01:00:00 2022-09-16 04:21:00    13103
1077     8623 2022-09-21 03:52:00 2022-09-23 23:23:00    13109
1078     8629 2022-09-29 22:37:00 2022-10-01 17:58:00    13111

[1079 rows x 4 columns]


In [16]:
# 生成正负标签
import json
from datetime import datetime
import pandas as pd 

# 读取 JSON 文件
with open("./data/sample_list.json", 'r') as f:
    all_sharp_list = json.load(f)

# 转换为DataFrame
time_df = pd.read_csv("./data/operational_form.csv")

import re

# 处理字典转换为DataFrame
flat_data = []

for entry in all_sharp_list:
    harp_num = entry['sharp_num']
    date_time = entry['date_time']
    folder_name = entry['folder_name']
    file_name = entry['file_name']
    
    flat_data.append({
        'HARPNUM': harp_num,
        'date_time': date_time,
        'folder_name': folder_name,
        'file_name': file_name
    })

# 创建DataFrame
event_df = pd.DataFrame(flat_data)
event_df["HARPNUM"] = event_df['HARPNUM'].astype(int)
event_df.head

event_df['date_time'] = event_df['file_name'].apply(lambda x: pd.to_datetime(re.search(r'\.(\d{8}_\d{6})_TAI$', x).group(1), format='%Y%m%d_%H%M%S'))

# 筛选出2022年10月1日之前的数据
event_df = event_df[event_df["date_time"] < datetime(2022, 10, 1)]

# 转换 time_df 的时间格式
time_df['Interval Start'] = pd.to_datetime(time_df['Interval Start'])
time_df['Interval End'] = pd.to_datetime(time_df['Interval End'])

# 生成label
def generate_labels(event_df, time_df):
    # 初始化一个新的label列，默认值为0
    result = []

    # 遍历event_df中的每一行数据
    for _, row in event_df.iterrows():
        harpnum = row['HARPNUM']  # 当前行的HARPNUM
        date_time = row['date_time']  # 当前行的date_time
        file_name = row["file_name"]
        
        # 如果harpnum不在time_df1和time_df2中的任何一个里面，标记为0
        if harpnum not in time_df['HARPNUM'].values:
            label = 0
        else:
            
            valid_intervals_df = time_df[time_df['HARPNUM'] == harpnum]
            if not valid_intervals_df[(valid_intervals_df['Interval Start'] <= date_time) & (valid_intervals_df['Interval End'] >= date_time)].empty:
                label = 1 # 如果date_time在time_df1的一个时间区间中，则label = 1, 即正样本
            else:
                label = 0
        
        result.append({
            "HARPNUM": harpnum,
            "file_name": file_name,
            "label": label
        })

    result = pd.DataFrame(result)
    print(result.value_counts(["label"]))
    
    return result

label_df = generate_labels(event_df, time_df)

label_df.to_csv('./data/label.csv', index = False)

print(label_df.value_counts(["label"]))


label
0        15566
1        14758
dtype: int64
label
0        15566
1        14758
dtype: int64


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 读取CSV文件
label_df = pd.read_csv('./data/label.csv')
label_df = label_df.copy()

# 设置随机种子
random_seed = 42

# 第一步：按8:1:1比例划分harpnum
harpnums = label_df['HARPNUM'].unique()
train_harpnums, valid_harpnums = train_test_split(harpnums, test_size=0.2, random_state=random_seed)
valid_harpnums, test_harpnums = train_test_split(valid_harpnums, test_size=0.5, random_state=random_seed)

# 使用划分后的harpnum筛选数据
sub_df = label_df[label_df['HARPNUM'].isin(train_harpnums)] 
valid1_df = label_df[label_df['HARPNUM'].isin(valid_harpnums)] # 20% * 50% 作为验证集1
test1_df = label_df[label_df['HARPNUM'].isin(test_harpnums)] # 20% * 50% 作为测试集1

# 第二步：对sub_df按6:1:1比例划分训练集、验证集、测试集
train_df, valid2_df = train_test_split(sub_df, test_size=0.25, random_state=random_seed)  # 80% * 75% 作为训练集
valid2_df, test2_df = train_test_split(valid2_df, test_size=0.5, random_state=random_seed)  # 80% * 25% * 50% 作为验证集2和测试集2

# 合并验证集
valid_df = pd.concat([valid1_df, valid2_df])

# 合并测试集
test_df = pd.concat([test1_df, test2_df])

train_df.to_csv("./data/split/train.csv")
valid_df.to_csv("./data/split/valid.csv")
test_df.to_csv("./data/split/test.csv")

valid1_df.to_csv("./data/split/valid1.csv")
valid2_df.to_csv("./data/split/valid2.csv")
test1_df.to_csv("./data/split/test1.csv")
test2_df.to_csv("./data/split/test2.csv")

# 展示数据集正负样本量
def print_class_distribution(df, name):
    print(f"{name} Class Distribution:")
    print(df['label'].value_counts())
    print()

print_class_distribution(train_df, "Train DataFrame")
print_class_distribution(valid1_df, "Validation1 DataFrame")
print_class_distribution(valid2_df, "Validation2 DataFrame")
print_class_distribution(test1_df, "Test1 DataFrame")
print_class_distribution(test2_df, "Test2 DataFrame")

Train DataFrame Class Distribution:
0    9416
1    9127
Name: label, dtype: int64

Validation1 DataFrame Class Distribution:
0    1454
1     976
Name: label, dtype: int64

Validation2 DataFrame Class Distribution:
0    1552
1    1539
Name: label, dtype: int64

Test1 DataFrame Class Distribution:
0    1587
1    1507
Name: label, dtype: int64

Test2 DataFrame Class Distribution:
1    1552
0    1539
Name: label, dtype: int64



In [ ]:
# 提前读取数据，并保存于 /data_work/data_work1/cnn_data 中

import os
import pandas as pd
import numpy as np
from astropy.io import fits
from scipy.ndimage import zoom
from IPython.display import clear_output

# 从 FITS 文件中读取数据
def get_fits_data(file_path):
    with fits.open(file_path) as hdu:
        data = hdu[1].data.astype(np.float32)
    return data

# 1. 数据处理和变换
def pad_matrix_edge(A):
    x, y = A.shape
    n = max(x, y)  # 新矩阵的大小
    
    # 创建一个大小为 n x n 的矩阵 B，并用0填充
    B = np.zeros((n, n), dtype=A.dtype)
    
    # 将原矩阵 A 的元素填充到 B 的中心
    start_x = (n - x) // 2  # 计算要填充的开始行
    start_y = (n - y) // 2  # 计算要填充的开始列
    
    B[start_x:start_x + x, start_y:start_y + y] = A
    
    return B

def Resizefits(fits_data):
    # 假设sample是numpy数组
    h, w = fits_data.shape
    max_dim = max(h, w)
    padded_sample = pad_matrix_edge(fits_data)

    # 计算变换系数
    resize_factors = 512 / max_dim

    # Resize到512x512
    resized_sample = zoom(padded_sample, (resize_factors, resize_factors), order=1)
    
    return resized_sample

# 读取 CSV 文件
def preprocess_fits_data(csv_file, data_path, save_path):
    # 创建保存目录
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    
    # 读取 CSV 文件
    df = pd.read_csv(csv_file)

    error_log = []
    
    # 遍历每一行
    for idx, row in df.iterrows():
        if idx % 10 == 0:
            clear_output(wait=True)

        harpnum = row['HARPNUM']
        file_name = row['file_name']
        print(f'{file_name} shart')
        
        # 构造 FITS 文件的路径
        try:
            blos_path = os.path.join(data_path, f'sharp_{harpnum:05d}', f'{file_name}.magnetogram.fits')
            br_path = os.path.join(data_path, f'sharp_{harpnum:05d}', f'{file_name}.Br.fits')
            bp_path = os.path.join(data_path, f'sharp_{harpnum:05d}', f'{file_name}.Bp.fits')
            bt_path = os.path.join(data_path, f'sharp_{harpnum:05d}', f'{file_name}.Bt.fits')
            
            # 读取 FITS 数据, 检查是否由
            blos_data = get_fits_data(blos_path)
            br_data = get_fits_data(br_path)
            bp_data = get_fits_data(bp_path)
            bt_data = get_fits_data(bt_path)

            if any(np.isnan(arr).any() for arr in [blos_data, br_data, bp_data, bt_data]):
                error_log.append(file_name)
                continue

            blos_data0 = Resizefits(blos_data)
            br_data0 = Resizefits(br_data)
            bp_data0 = Resizefits(bp_data)
            bt_data0 = Resizefits(bt_data)
            data_processed = np.stack([blos_data0, br_data0, bp_data0, bt_data0], axis=-1)
            if np.any(np.isnan(data_processed)):
                error_log.append(file_name)
                continue
            
            # 保存为 .npy 格式
            save_file = os.path.join(save_path, f'{file_name}.data.npy')
            np.save(save_file, data_processed)
            
            print(f"Processed and saved: {save_file}")
        except:
            error_log.append(file_name)

    return error_log

# 使用示例
csv_file = './data/label.csv'  # CSV 文件路径
data_path = '/data_work/data_sharp_cea/'    # FITS 文件存储路径
save_path = '/data_work/cnn_data/'           # 存储 .npy 文件的目标路径

error_log = preprocess_fits_data(csv_file, data_path, save_path)

from pprint import pprint 
pprint(error_log)

hmi.sharp_cea_720s.8634.20220930_080000_TAI shart
Processed and saved: /data1/data_zz/data_work1/cnn_data/hmi.sharp_cea_720s.8634.20220930_080000_TAI.data.npy
hmi.sharp_cea_720s.8636.20220930_080000_TAI shart
Processed and saved: /data1/data_zz/data_work1/cnn_data/hmi.sharp_cea_720s.8636.20220930_080000_TAI.data.npy
hmi.sharp_cea_720s.8641.20220930_100000_TAI shart
Processed and saved: /data1/data_zz/data_work1/cnn_data/hmi.sharp_cea_720s.8641.20220930_100000_TAI.data.npy
hmi.sharp_cea_720s.8642.20220930_220000_TAI shart
Processed and saved: /data1/data_zz/data_work1/cnn_data/hmi.sharp_cea_720s.8642.20220930_220000_TAI.data.npy
['hmi.sharp_cea_720s.4000.20140416_040000_TAI',
 'hmi.sharp_cea_720s.4000.20140417_080000_TAI',
 'hmi.sharp_cea_720s.4000.20140418_080000_TAI',
 'hmi.sharp_cea_720s.4000.20140419_080000_TAI',
 'hmi.sharp_cea_720s.4000.20140420_080000_TAI',
 'hmi.sharp_cea_720s.4000.20140421_080000_TAI',
 'hmi.sharp_cea_720s.4000.20140422_010000_TAI',
 'hmi.sharp_cea_720s.5500.20